# Exploración de Forecast por SKU

Este notebook muestra cómo usar el pipeline de forecasting sobre un SKU individual:
1. Clasificar el SKU
2. Ver la serie de demanda limpia
3. Correr el selector automático (backtest + horse-race)
4. Visualizar el forecast resultante

> **Restart & Run All antes de usar.**

In [1]:
# ── Celda 1: setup + imports ───────────────────────────────────────────────
import subprocess, sys, importlib, pathlib, math

REPO_ROOT = pathlib.Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

def _ensure(import_name: str, pip_spec: str):
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f'Instalando {pip_spec} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pip_spec])
        print(f'  {pip_spec} instalado.')

_ensure('plotly',        'plotly>=5.24')
_ensure('statsforecast', 'statsforecast>=1.7')

try:
    import planning_core  # noqa: F401
except ImportError:
    print('Instalando sota-simulated-planning en modo editable ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(REPO_ROOT)])

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'notebook_connected'

from planning_core.repository import CanonicalRepository
from planning_core.services import PlanningService
from planning_core.forecasting.backtest import backtest_summary

repo = CanonicalRepository()
service = PlanningService(repo)

overview = service.dataset_overview()
print(f"SKUs        : {overview['sku_count']}")
print(f"Locations   : {overview['location_count']}")
print(f"Central     : {overview['central_location']}")
print(f"Periodo     : {overview['date_range']['start']} → {overview['date_range']['end']}")

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SKUs        : 800
Locations   : 6
Central     : CD Santiago
Periodo     : 2022-01-01 → 2024-12-31


## 1. Seleccionar SKU y revisar clasificación

In [8]:
SKU = 'SKU-00020'  # ← cambiar por cualquier SKU del catálogo

profile = service.classify_single_sku(SKU)
if profile is None:
    raise ValueError(f'SKU no encontrado o sin transacciones: {SKU}')

# ABC requiere el catálogo completo para calcular percentiles de revenue.
# Se hace un lookup puntual sobre classify_catalog (lento la primera vez, cacheado después).
catalog_df = service.classify_catalog(granularity=profile['granularity'])
sku_row    = catalog_df[catalog_df['sku'] == SKU]
abc_class  = sku_row['abc_class'].iloc[0] if not sku_row.empty else 'N/A'

print(f"SB class    : {profile['sb_class']}")
print(f"ABC         : {abc_class}")
print(f"XYZ         : {profile['xyz_class']}")
print(f"Lifecycle   : {profile['lifecycle']}")
print(f"Seasonal    : {profile['is_seasonal']}")
print(f"Quality     : {profile['quality_score']:.2f}")
print(f"Periodos    : {profile['total_periods']}")

SB class    : smooth
ABC         : C
XYZ         : X
Lifecycle   : mature
Seasonal    : False
Quality     : 0.93
Periodos    : 36


## 2. Serie de demanda limpia (outliers tratados)

In [3]:
clean_df = service.sku_clean_series(SKU, granularity='M')
print(clean_df.head(10))
print(f"\nPeriodos totales : {len(clean_df)}")
print(f"Outliers         : {clean_df['is_outlier'].sum()}")

      period  demand  is_outlier  demand_clean
0 2022-01-01    20.0       False          20.0
1 2022-02-01    22.0       False          22.0
2 2022-03-01    22.0       False          22.0
3 2022-04-01    18.0       False          18.0
4 2022-05-01    23.0       False          23.0
5 2022-06-01    27.0       False          27.0
6 2022-07-01    17.0       False          17.0
7 2022-08-01    29.0       False          29.0
8 2022-09-01    21.0       False          21.0
9 2022-10-01    37.0        True          35.0

Periodos totales : 36
Outliers         : 2


## 3. Correr el selector automático (backtest + horse-race)

In [4]:
result = service.sku_forecast(SKU, granularity='M', h=6, n_windows=3)

mase_val = result.get('mase')
mase_str = f'{mase_val:.3f}' if mase_val is not None and not math.isnan(mase_val) else 'N/A (fallback sin backtest)'

print(f"Status      : {result['status']}")
print(f"Modelo      : {result['model']}")
print(f"MASE        : {mase_str}")
print()
print(result['forecast'])

Status      : ok
Modelo      : AutoARIMA
MASE        : 0.804

          ds       yhat  yhat_lo80  yhat_hi80
0 2025-01-01  21.805556  13.535146  30.075965
1 2025-02-01  21.805556  13.535146  30.075965
2 2025-03-01  21.805556  13.535146  30.075965
3 2025-04-01  21.805556  13.535146  30.075965
4 2025-05-01  21.805556  13.535146  30.075965
5 2025-06-01  21.805556  13.535146  30.075965


## 4. Comparar candidatos del horse-race

In [5]:
summary = backtest_summary(result['backtest'])
print(summary[['model', 'mase', 'wape', 'bias', 'n_windows', 'status']].to_string(index=False))

        model     mase     wape      bias  n_windows status
    AutoARIMA 0.804316 0.239901 -0.023099          3     ok
      AutoETS 0.804629 0.239988 -0.023269          3     ok
SeasonalNaive 1.175341 0.328990 -0.014904          3     ok
     LightGBM      NaN      NaN       NaN          0  error


## 5. Visualizar forecast vs histórico

In [7]:
hist = clean_df[['period', 'demand_clean']].rename(columns={'period': 'ds', 'demand_clean': 'y'})
fc   = result['forecast']

mase_val   = result.get('mase')
mase_label = f'{mase_val:.2f}' if mase_val is not None and not math.isnan(mase_val) else 'fallback'

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hist['ds'], y=hist['y'],
    name='Demanda histórica', line=dict(color='#2980b9')
))
fig.add_trace(go.Scatter(
    x=fc['ds'], y=fc['yhat'],
    name=f"Forecast ({result['model']})",
    line=dict(color='#e74c3c', dash='dash')
))
if 'yhat_lo80' in fc.columns:
    fig.add_trace(go.Scatter(
        x=list(fc['ds']) + list(fc['ds'][::-1]),
        y=list(fc['yhat_hi80']) + list(fc['yhat_lo80'][::-1]),
        fill='toself', fillcolor='rgba(231,76,60,0.15)',
        line=dict(color='rgba(0,0,0,0)'), name='IC 80%'
    ))

fig.update_layout(
    title=f'{SKU} — {result["model"]}  (MASE={mase_label})',
    xaxis_title='Periodo', yaxis_title='Demanda',
    template='plotly_white', height=420
)
fig.show()